In [2]:
!pip install polars plotly dash dash-bootstrap-components

In [3]:
"""
NYC Restaurant Inspection Dashboard
=====================================
Requirements: pip install polars plotly dash dash-bootstrap-components
Usage:        python nyc_inspection_dashboard.py
              Then open http://127.0.0.1:8050 in your browser
              A standalone HTML export is also saved as dashboard_export.html
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, sys, warnings
warnings.filterwarnings("ignore")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import dash
from dash import dcc, html, Input, Output, callback
import dash_bootstrap_components as dbc

# ──────────────────────────────────────────────────────────────────────────────
# 1. CONSTANTS & THEME
# ──────────────────────────────────────────────────────────────────────────────
CSV_PATH = "/content/nyc_restaurant_inspections_CLEAN.csv"

THEME = dict(
    bg         = "#0A0E1A",
    surface    = "#111827",
    surface2   = "#1C2333",
    border     = "#2A3347",
    accent     = "#00D4FF",
    accent2    = "#FF6B6B",
    accent3    = "#4ECDC4",
    accent4    = "#FFE66D",
    accent5    = "#A78BFA",
    text_pri   = "#F1F5F9",
    text_sec   = "#94A3B8",
    font       = "DM Mono, monospace",
)

PLOTLY_LAYOUT = dict(
    paper_bgcolor = "rgba(0,0,0,0)",
    plot_bgcolor  = "rgba(0,0,0,0)",
    font          = dict(family=THEME["font"], color=THEME["text_sec"], size=11),
    title_font    = dict(family=THEME["font"], color=THEME["text_pri"], size=14),
    margin        = dict(l=10, r=10, t=40, b=10),
    legend        = dict(
        bgcolor      = "rgba(0,0,0,0)",
        bordercolor  = THEME["border"],
        borderwidth  = 1,
        font         = dict(color=THEME["text_sec"], size=10),
    ),
    colorway = [
        THEME["accent"], THEME["accent2"], THEME["accent3"],
        THEME["accent4"], THEME["accent5"], "#F97316", "#10B981",
        "#EC4899", "#6366F1", "#84CC16",
    ],
)

AXIS_STYLE = dict(
    showgrid       = True,
    gridcolor      = THEME["border"],
    gridwidth      = 0.5,
    zeroline       = False,
    tickfont       = dict(color=THEME["text_sec"], size=10),
    title_font     = dict(color=THEME["text_sec"]),
    linecolor      = THEME["border"],
    showline       = True,
)

# ──────────────────────────────────────────────────────────────────────────────
# 2. DATA LOADING & CLEANING  (Polars, lazy where possible)
# ──────────────────────────────────────────────────────────────────────────────

def load_and_clean(path: str) -> pl.DataFrame:
    """
    Load CSV with Polars lazy scan, clean, engineer features, collect.
    Falls back gracefully if file is not found (uses embedded sample data).
    """
    try:
        lf = pl.scan_csv(
            path,
            infer_schema_length=10_000,
            ignore_errors=True,
            null_values=["", "N/A", "NA", "NULL", "null"],
            encoding="utf8-lossy",
        )
    except Exception:
        print(f"[WARN] Could not read '{path}'. Using bundled sample data.")
        lf = _load_sample()

    lf = (
        lf
        # ── normalise column names ──────────────────────────────────────────
        .rename({c: c.strip().upper().replace(" ", "_") for c in lf.columns})
        # ── parse date ─────────────────────────────────────────────────────
        .with_columns(
            pl.col("INSPECTION_DATE").str.to_date(format="%Y-%m-%d", strict=False)
              .alias("INSPECTION_DATE"),
        )
        # ── drop rows missing key fields ───────────────────────────────────
        .filter(
            pl.col("BORO").is_not_null()
            & pl.col("BORO").ne("0")
            & pl.col("BORO").ne("")
            & pl.col("INSPECTION_DATE").is_not_null()
        )
        # ── clean BORO casing ──────────────────────────────────────────────
        .with_columns(
            pl.col("BORO").str.to_titlecase().alias("BORO"),
        )
        # ── numeric score ──────────────────────────────────────────────────
        .with_columns(
            pl.col("SCORE").cast(pl.Float32, strict=False).alias("SCORE"),
        )
        # ── grade: fill blank/null with "Pending" ─────────────────────────
        .with_columns(
            pl.when(pl.col("GRADE").is_null() | pl.col("GRADE").str.strip_chars().eq(""))
              .then(pl.lit("Pending"))
              .otherwise(pl.col("GRADE").str.strip_chars())
              .alias("GRADE")
        )
        # ── is_critical feature ────────────────────────────────────────────
        .with_columns(
            pl.when(
                pl.col("CRITICAL_FLAG").str.to_lowercase().str.contains("critical")
                & ~pl.col("CRITICAL_FLAG").str.to_lowercase().str.contains("not")
            )
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .cast(pl.Int8)
            .alias("IS_CRITICAL"),
        )
        # ── risk score ─────────────────────────────────────────────────────
        .with_columns(
            (pl.col("SCORE").fill_null(0) + pl.col("IS_CRITICAL") * 10)
            .alias("RISK_SCORE"),
        )
        # ── date features ──────────────────────────────────────────────────
        .with_columns(
            pl.col("INSPECTION_DATE").dt.year().alias("YEAR"),
            pl.col("INSPECTION_DATE").dt.month().alias("MONTH"),
            pl.col("INSPECTION_DATE").dt.strftime("%Y-%m").alias("YEAR_MONTH"),
        )
        # ── clean cuisine ──────────────────────────────────────────────────
        .with_columns(
            pl.col("CUISINE_DESCRIPTION")
              .str.strip_chars()
              .fill_null("Other")
              .alias("CUISINE_DESCRIPTION"),
        )
    )

    df = lf.collect()
    return df


def _load_sample() -> pl.LazyFrame:
    """Return a LazyFrame from the embedded sample rows."""
    import io
    RAW = """CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,INSPECTION TYPE,IS_SENTINEL_DATE,SCORE_INVALID,SCORE_EXTREME,HAS_GRADE,VIOLATION_MISMATCH
40511702,NOTARO RESTAURANT,MANHATTAN,635,SECOND AVENUE,10016,2126863400,Italian,2015-06-15,Violations were cited in the following area(s).,02B,Hot food item not held at or above 140F.,Critical,30.0,,,Cycle Inspection / Initial Inspection,False,False,False,False,False
50046354,VITE BAR,QUEENS,2507,BROADWAY,11106,3478134702,Italian,2016-10-03,Violations were cited in the following area(s).,10F,Non-food contact surface improperly constructed.,Not Critical,2.0,,,Pre-permit (Operational) / Initial Inspection,False,False,False,False,False
50061389,TACKS CHINESE TAKE OUT,STATEN ISLAND,11C,HOLDEN BLVD,10314,7189839854,Chinese,2017-05-17,Violations were cited in the following area(s).,02G,Cold food item held above 41F.,Critical,46.0,,,Pre-permit (Operational) / Initial Inspection,False,False,False,False,False
41516263,NO QUARTER,BROOKLYN,8015,5 AVENUE,11209,7187019180,American,2017-03-30,Violations were cited in the following area(s).,04M,Live roaches present.,Critical,18.0,,,Cycle Inspection / Initial Inspection,False,False,False,False,False
40807238,RICHMOND COUNTY COUNTRY CLUB,STATEN ISLAND,1122,TODT HILL ROAD,10304,7183510600,American,2017-06-14,Violations were cited in the following area(s).,02G,Cold food item held above 41F.,Critical,12.0,A,2017-06-14,Cycle Inspection / Initial Inspection,False,False,False,True,False
40376944,TOMOE SUSHI,MANHATTAN,172,THOMPSON STREET,10012,2127779346,Japanese,2015-10-06,Violations were cited in the following area(s).,02G,Cold food item held above 41F.,Critical,13.0,A,2015-10-06,Cycle Inspection / Re-inspection,False,False,False,True,False
41552184,NOM WAH TEA PALOR,MANHATTAN,13,DOYERS STREET,10013,2129626047,Chinese,2014-09-02,Violations were cited in the following area(s).,02G,Cold food item held above 41F.,Critical,18.0,B,2014-09-02,Cycle Inspection / Re-inspection,False,False,False,True,False
41696159,LUIGIS PIZZA,BRONX,119,EAST MOUNT EDEN AVENUE,10452,7182941800,Pizza,2014-10-27,Violations were cited in the following area(s).,10B,Plumbing not properly installed.,Not Critical,13.0,A,2014-10-27,Cycle Inspection / Re-inspection,False,False,False,True,False"""
    return pl.read_csv(io.StringIO(RAW), infer_schema_length=100, null_values=["", "N/A"]).lazy()


# ──────────────────────────────────────────────────────────────────────────────
# 3. PRE-AGGREGATIONS  (computed once, reused by callbacks)
# ──────────────────────────────────────────────────────────────────────────────

def precompute(df: pl.DataFrame) -> dict:
    agg = {}

    # borough counts
    agg["boro_counts"] = (
        df.group_by("BORO")
          .agg(pl.len().alias("COUNT"))
          .sort("COUNT", descending=True)
    )

    # top 10 cuisines
    agg["cuisine_top10"] = (
        df.group_by("CUISINE_DESCRIPTION")
          .agg(pl.len().alias("COUNT"))
          .sort("COUNT", descending=True)
          .head(10)
    )

    # inspections over time (monthly)
    agg["monthly"] = (
        df.group_by("YEAR_MONTH")
          .agg(pl.len().alias("COUNT"))
          .sort("YEAR_MONTH")
    )

    # score distribution (pre-bin with polars)
    valid_scores = df.filter(pl.col("SCORE").is_not_null())
    agg["scores"] = valid_scores["SCORE"].to_list()

    # grade distribution
    agg["grade_dist"] = (
        df.filter(pl.col("GRADE").is_in(["A", "B", "C", "Z", "P", "Pending"]))
          .group_by("GRADE")
          .agg(pl.len().alias("COUNT"))
          .sort("COUNT", descending=True)
    )

    # top violation codes
    agg["top_violations"] = (
        df.filter(pl.col("VIOLATION_CODE").is_not_null())
          .group_by("VIOLATION_CODE")
          .agg(pl.len().alias("COUNT"))
          .sort("COUNT", descending=True)
          .head(12)
    )

    # critical vs non-critical by borough
    agg["critical_by_boro"] = (
        df.group_by(["BORO", "IS_CRITICAL"])
          .agg(pl.len().alias("COUNT"))
          .sort(["BORO", "IS_CRITICAL"])
    )

    # heatmap: borough x grade
    valid_grades = ["A", "B", "C", "Z", "P"]
    agg["heatmap_df"] = (
        df.filter(pl.col("GRADE").is_in(valid_grades))
          .group_by(["BORO", "GRADE"])
          .agg(pl.len().alias("COUNT"))
    )

    return agg


def compute_kpis(df: pl.DataFrame) -> dict:
    total = len(df)
    avg_score = df["SCORE"].drop_nulls().mean()
    pct_critical = df["IS_CRITICAL"].mean() * 100
    grade_a_pct = (
        df.filter(pl.col("GRADE") == "A").shape[0] / max(total, 1) * 100
    )
    return dict(
        total        = f"{total:,}",
        avg_score    = f"{avg_score:.1f}" if avg_score else "N/A",
        pct_critical = f"{pct_critical:.1f}%",
        grade_a_pct  = f"{grade_a_pct:.1f}%",
    )


# ──────────────────────────────────────────────────────────────────────────────
# 4. CHART FACTORIES
# ──────────────────────────────────────────────────────────────────────────────

def _apply_layout(fig, title="", height=320):
    fig.update_layout(
        **PLOTLY_LAYOUT,
        title_text=title,
        height=height,
        xaxis=AXIS_STYLE,
        yaxis=AXIS_STYLE,
    )
    return fig


def fig_boro_bar(agg: dict) -> go.Figure:
    df = agg["boro_counts"]
    fig = go.Figure(go.Bar(
        x=df["BORO"].to_list(),
        y=df["COUNT"].to_list(),
        marker=dict(
            color=df["COUNT"].to_list(),
            colorscale=[[0, "#1C2333"], [1, THEME["accent"]]],
            showscale=False,
            line=dict(color=THEME["accent"], width=0.5),
        ),
        hovertemplate="<b>%{x}</b><br>Inspections: %{y:,}<extra></extra>",
    ))
    _apply_layout(fig, "Inspections by Borough")
    return fig


def fig_cuisine_bar(agg: dict) -> go.Figure:
    df = agg["cuisine_top10"]
    names = df["CUISINE_DESCRIPTION"].to_list()
    counts = df["COUNT"].to_list()
    fig = go.Figure(go.Bar(
        x=counts,
        y=names,
        orientation="h",
        marker=dict(
            color=counts,
            colorscale=[[0, "#1C2333"], [1, THEME["accent3"]]],
            showscale=False,
            line=dict(color=THEME["accent3"], width=0.5),
        ),
        hovertemplate="<b>%{y}</b><br>Count: %{x:,}<extra></extra>",
    ))
    _apply_layout(fig, "Top 10 Cuisines")
    fig.update_layout(yaxis=dict(**AXIS_STYLE, autorange="reversed"))
    return fig


def fig_monthly_line(agg: dict) -> go.Figure:
    df = agg["monthly"]
    x = df["YEAR_MONTH"].to_list()
    y = df["COUNT"].to_list()
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode="lines",
        line=dict(color=THEME["accent4"], width=2, shape="spline"),
        fill="tozeroy",
        fillcolor="rgba(255,230,109,0.07)",
        hovertemplate="<b>%{x}</b><br>Inspections: %{y:,}<extra></extra>",
    ))
    _apply_layout(fig, "Inspections Over Time (Monthly)")
    return fig


def fig_score_histogram(agg: dict) -> go.Figure:
    fig = go.Figure(go.Histogram(
        x=agg["scores"],
        nbinsx=40,
        marker=dict(
            color=THEME["accent5"],
            opacity=0.8,
            line=dict(color=THEME["bg"], width=0.3),
        ),
        hovertemplate="Score Range: %{x}<br>Count: %{y:,}<extra></extra>",
    ))
    _apply_layout(fig, "Score Distribution")
    return fig


def fig_grade_pie(agg: dict) -> go.Figure:
    df = agg["grade_dist"]
    grade_colors = {
        "A": THEME["accent3"],  "B": THEME["accent4"],
        "C": THEME["accent2"],  "Z": THEME["accent5"],
        "P": "#F97316",         "Pending": THEME["text_sec"],
    }
    labels = df["GRADE"].to_list()
    values = df["COUNT"].to_list()
    colors = [grade_colors.get(g, THEME["text_sec"]) for g in labels]
    fig = go.Figure(go.Pie(
        labels=labels,
        values=values,
        hole=0.55,
        marker=dict(colors=colors, line=dict(color=THEME["bg"], width=2)),
        hovertemplate="<b>Grade %{label}</b><br>Count: %{value:,}<br>%{percent}<extra></extra>",
        textfont=dict(color=THEME["text_pri"], size=11),
    ))
    fig.update_layout(
        **PLOTLY_LAYOUT,
        title_text="Grade Distribution",
        height=320,
        annotations=[dict(
            text="Grades", x=0.5, y=0.5,
            font_size=13, font_color=THEME["text_sec"],
            showarrow=False
        )],
    )
    return fig


def fig_violation_bar(agg: dict) -> go.Figure:
    df = agg["top_violations"]
    codes  = df["VIOLATION_CODE"].to_list()
    counts = df["COUNT"].to_list()
    fig = go.Figure(go.Bar(
        x=counts,
        y=codes,
        orientation="h",
        marker=dict(
            color=counts,
            colorscale=[[0, "#1C2333"], [1, THEME["accent2"]]],
            showscale=False,
            line=dict(color=THEME["accent2"], width=0.5),
        ),
        hovertemplate="<b>Code %{y}</b><br>Count: %{x:,}<extra></extra>",
    ))
    _apply_layout(fig, "Top Violation Codes")
    fig.update_layout(yaxis=dict(**AXIS_STYLE, autorange="reversed"))
    return fig


def fig_critical_comparison(agg: dict) -> go.Figure:
    df = agg["critical_by_boro"]
    boros     = sorted(df["BORO"].unique().to_list())
    critical  = []
    noncrit   = []
    for b in boros:
        sub = df.filter(pl.col("BORO") == b)
        c = sub.filter(pl.col("IS_CRITICAL") == 1)["COUNT"].sum() if sub.filter(pl.col("IS_CRITICAL") == 1).shape[0] else 0
        n = sub.filter(pl.col("IS_CRITICAL") == 0)["COUNT"].sum() if sub.filter(pl.col("IS_CRITICAL") == 0).shape[0] else 0
        critical.append(c)
        noncrit.append(n)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Critical",
        x=boros, y=critical,
        marker=dict(color=THEME["accent2"], line=dict(color=THEME["bg"], width=0.5)),
        hovertemplate="<b>%{x}</b><br>Critical: %{y:,}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name="Non-Critical",
        x=boros, y=noncrit,
        marker=dict(color=THEME["accent3"], line=dict(color=THEME["bg"], width=0.5)),
        hovertemplate="<b>%{x}</b><br>Non-Critical: %{y:,}<extra></extra>",
    ))
    _apply_layout(fig, "Critical vs Non-Critical by Borough")
    fig.update_layout(barmode="group")
    return fig


def fig_heatmap(agg: dict) -> go.Figure:
    df = agg["heatmap_df"]
    boros  = sorted(df["BORO"].unique().to_list())
    grades = ["A", "B", "C", "Z", "P"]
    z = []
    for b in boros:
        row = []
        for g in grades:
            sub = df.filter((pl.col("BORO") == b) & (pl.col("GRADE") == g))
            row.append(sub["COUNT"].sum() if sub.shape[0] > 0 else 0)
        z.append(row)

    fig = go.Figure(go.Heatmap(
        z=z,
        x=grades,
        y=boros,
        colorscale=[[0, THEME["bg"]], [0.5, "#1E3A5F"], [1, THEME["accent"]]],
        hovertemplate="<b>%{y}</b> — Grade <b>%{x}</b><br>Count: %{z:,}<extra></extra>",
        showscale=True,
        colorbar=dict(
            tickfont=dict(color=THEME["text_sec"], size=9),
            bgcolor="rgba(0,0,0,0)",
            bordercolor=THEME["border"],
            borderwidth=1,
        ),
    ))
    _apply_layout(fig, "Borough × Grade Heatmap", height=320)
    fig.update_layout(xaxis=dict(**AXIS_STYLE), yaxis=dict(**AXIS_STYLE))
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 5. LAYOUT HELPERS
# ──────────────────────────────────────────────────────────────────────────────

def kpi_card(label: str, value: str, accent: str) -> html.Div:
    return html.Div(
        className="kpi-card",
        style={"borderTopColor": accent},
        children=[
            html.Div(value, className="kpi-value", style={"color": accent}),
            html.Div(label, className="kpi-label"),
        ],
    )


CUSTOM_CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Outfit:wght@300;400;600;700&display=swap');

*, *::before, *::after {{ box-sizing: border-box; }}

body {{
    margin: 0;
    background: {THEME["bg"]};
    font-family: 'Outfit', sans-serif;
    color: {THEME["text_pri"]};
}}

.dashboard-wrapper {{
    min-height: 100vh;
    padding: 0 0 40px 0;
    background:
        radial-gradient(ellipse at 10% 0%, rgba(0,212,255,0.06) 0%, transparent 60%),
        radial-gradient(ellipse at 90% 100%, rgba(167,139,250,0.05) 0%, transparent 60%),
        {THEME["bg"]};
}}

/* ── HEADER ── */
.header {{
    background: linear-gradient(135deg, #0D1420 0%, #111827 100%);
    border-bottom: 1px solid {THEME["border"]};
    padding: 22px 32px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    position: sticky;
    top: 0;
    z-index: 100;
    backdrop-filter: blur(10px);
}}
.header-logo {{
    display: flex;
    align-items: center;
    gap: 12px;
}}
.header-dot {{
    width: 10px;
    height: 10px;
    border-radius: 50%;
    background: {THEME["accent"]};
    box-shadow: 0 0 12px {THEME["accent"]};
    animation: pulse 2s infinite;
}}
@keyframes pulse {{
    0%,100% {{ opacity: 1; }}
    50% {{ opacity: 0.4; }}
}}
.header-title {{
    font-family: 'DM Mono', monospace;
    font-size: 18px;
    font-weight: 500;
    color: {THEME["text_pri"]};
    letter-spacing: 0.04em;
}}
.header-subtitle {{
    font-size: 11px;
    color: {THEME["text_sec"]};
    letter-spacing: 0.1em;
    text-transform: uppercase;
    font-family: 'DM Mono', monospace;
}}
.header-badge {{
    background: rgba(0,212,255,0.1);
    border: 1px solid rgba(0,212,255,0.3);
    border-radius: 6px;
    padding: 6px 14px;
    font-family: 'DM Mono', monospace;
    font-size: 11px;
    color: {THEME["accent"]};
    letter-spacing: 0.06em;
}}

/* ── MAIN CONTENT ── */
.main-content {{
    padding: 24px 32px;
    max-width: 1600px;
    margin: 0 auto;
}}

/* ── FILTER PANEL ── */
.filter-panel {{
    background: {THEME["surface"]};
    border: 1px solid {THEME["border"]};
    border-radius: 12px;
    padding: 20px 24px;
    margin-bottom: 24px;
}}
.filter-panel-title {{
    font-family: 'DM Mono', monospace;
    font-size: 11px;
    color: {THEME["accent"]};
    text-transform: uppercase;
    letter-spacing: 0.12em;
    margin-bottom: 16px;
    display: flex;
    align-items: center;
    gap: 8px;
}}
.filter-panel-title::before {{
    content: '';
    display: inline-block;
    width: 4px;
    height: 4px;
    border-radius: 50%;
    background: {THEME["accent"]};
    box-shadow: 0 0 8px {THEME["accent"]};
}}
.filter-row {{
    display: grid;
    grid-template-columns: repeat(4, 1fr) 1.5fr;
    gap: 16px;
    align-items: start;
}}
.filter-label {{
    font-family: 'DM Mono', monospace;
    font-size: 10px;
    color: {THEME["text_sec"]};
    text-transform: uppercase;
    letter-spacing: 0.1em;
    margin-bottom: 6px;
}}

/* Dash dropdown override */
.Select-control {{
    background: {THEME["surface2"]} !important;
    border: 1px solid {THEME["border"]} !important;
    border-radius: 8px !important;
    color: {THEME["text_pri"]} !important;
}}
.Select-menu-outer {{
    background: {THEME["surface2"]} !important;
    border: 1px solid {THEME["border"]} !important;
    border-radius: 8px !important;
}}
.Select-option {{
    background: {THEME["surface2"]} !important;
    color: {THEME["text_sec"]} !important;
    font-size: 12px !important;
}}
.Select-option.is-focused {{
    background: rgba(0,212,255,0.1) !important;
    color: {THEME["accent"]} !important;
}}
.Select-value-label {{ color: {THEME["text_pri"]} !important; font-size: 12px !important; }}
.Select-placeholder {{ color: {THEME["text_sec"]} !important; font-size: 12px !important; }}
.VirtualizedSelectFocusedOption {{ background: rgba(0,212,255,0.1) !important; }}
.Select--multi .Select-value {{
    background: rgba(0,212,255,0.12) !important;
    border: 1px solid rgba(0,212,255,0.3) !important;
    color: {THEME["accent"]} !important;
    border-radius: 4px !important;
}}
.Select--multi .Select-value-icon:hover {{
    background: rgba(0,212,255,0.2) !important;
    color: {THEME["accent"]} !important;
}}

/* Range slider */
.rc-slider-rail {{ background: {THEME["surface2"]} !important; }}
.rc-slider-track {{ background: {THEME["accent"]} !important; }}
.rc-slider-handle {{ border-color: {THEME["accent"]} !important; background: {THEME["bg"]} !important; }}
.rc-slider-handle:hover {{ border-color: {THEME["accent"]} !important; box-shadow: 0 0 8px {THEME["accent"]} !important; }}

/* ── KPI CARDS ── */
.kpi-row {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 16px;
    margin-bottom: 24px;
}}
.kpi-card {{
    background: {THEME["surface"]};
    border: 1px solid {THEME["border"]};
    border-top: 3px solid;
    border-radius: 12px;
    padding: 20px 24px;
    transition: transform 0.2s, box-shadow 0.2s;
}}
.kpi-card:hover {{
    transform: translateY(-2px);
    box-shadow: 0 8px 24px rgba(0,0,0,0.3);
}}
.kpi-value {{
    font-family: 'DM Mono', monospace;
    font-size: 28px;
    font-weight: 500;
    letter-spacing: -0.02em;
    margin-bottom: 6px;
}}
.kpi-label {{
    font-size: 11px;
    color: {THEME["text_sec"]};
    text-transform: uppercase;
    letter-spacing: 0.1em;
    font-family: 'DM Mono', monospace;
}}

/* ── CHART CARDS ── */
.chart-card {{
    background: {THEME["surface"]};
    border: 1px solid {THEME["border"]};
    border-radius: 12px;
    padding: 8px 4px 4px 4px;
    overflow: hidden;
    transition: border-color 0.2s, box-shadow 0.2s;
}}
.chart-card:hover {{
    border-color: rgba(0,212,255,0.25);
    box-shadow: 0 4px 20px rgba(0,0,0,0.25);
}}

.charts-grid-2 {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
    margin-bottom: 16px;
}}
.charts-grid-3 {{
    display: grid;
    grid-template-columns: 1fr 1fr 1fr;
    gap: 16px;
    margin-bottom: 16px;
}}
.charts-grid-full {{
    margin-bottom: 16px;
}}

/* ── FOOTER ── */
.footer {{
    text-align: center;
    padding: 20px;
    font-family: 'DM Mono', monospace;
    font-size: 10px;
    color: {THEME["border"]};
    letter-spacing: 0.1em;
    text-transform: uppercase;
    border-top: 1px solid {THEME["border"]};
    margin-top: 24px;
}}

/* ── SCROLLBAR ── */
::-webkit-scrollbar {{ width: 6px; height: 6px; }}
::-webkit-scrollbar-track {{ background: {THEME["bg"]}; }}
::-webkit-scrollbar-thumb {{ background: {THEME["border"]}; border-radius: 3px; }}
::-webkit-scrollbar-thumb:hover {{ background: {THEME["text_sec"]}; }}
"""


# ──────────────────────────────────────────────────────────────────────────────
# 6. DASH APP SETUP
# ──────────────────────────────────────────────────────────────────────────────

print("[INFO] Loading data …")
DF_FULL = load_and_clean(CSV_PATH)
print(f"[INFO] Loaded {len(DF_FULL):,} rows")

# global pre-agg on full dataset (used as default)
AGG_FULL   = precompute(DF_FULL)
KPIS_FULL  = compute_kpis(DF_FULL)

BOROS      = sorted(DF_FULL["BORO"].drop_nulls().unique().to_list())
CUISINES   = (
    DF_FULL.group_by("CUISINE_DESCRIPTION")
           .agg(pl.len().alias("n"))
           .sort("n", descending=True)
           .head(60)
           ["CUISINE_DESCRIPTION"].to_list()
)
GRADES     = sorted(DF_FULL["GRADE"].drop_nulls().unique().to_list())
YEAR_MIN   = int(DF_FULL["YEAR"].drop_nulls().min())
YEAR_MAX   = int(DF_FULL["YEAR"].drop_nulls().max())


app = dash.Dash(
    __name__,
    suppress_callback_exceptions=True,
    title="NYC Restaurant Inspections",
    external_stylesheets=[],
)
app.index_string = f"""
<!DOCTYPE html>
<html>
  <head>
    {{%metas%}}
    <title>{{%title%}}</title>
    {{%favicon%}}
    {{%css%}}
    <style>{CUSTOM_CSS}</style>
  </head>
  <body>
    {{%app_entry%}}
    <footer>{{%config%}}{{%scripts%}}{{%renderer%}}</footer>
  </body>
</html>
"""

app.layout = html.Div(className="dashboard-wrapper", children=[

    # ── HEADER ──────────────────────────────────────────────────────────────
    html.Div(className="header", children=[
        html.Div(className="header-logo", children=[
            html.Div(className="header-dot"),
            html.Div([
                html.Div("NYC Restaurant Inspections", className="header-title"),
                html.Div("DOHMH · Health Inspection Analytics Platform", className="header-subtitle"),
            ]),
        ]),
        html.Div(id="header-badge", className="header-badge",
                 children=f"DATASET LOADED · {len(DF_FULL):,} RECORDS"),
    ]),

    # ── MAIN ────────────────────────────────────────────────────────────────
    html.Div(className="main-content", children=[

        # ── FILTERS ─────────────────────────────────────────────────────────
        html.Div(className="filter-panel", children=[
            html.Div("⚙  Active Filters", className="filter-panel-title"),
            html.Div(className="filter-row", children=[
                html.Div([
                    html.Div("Borough", className="filter-label"),
                    dcc.Dropdown(
                        id="filter-boro",
                        options=[{"label": b, "value": b} for b in BOROS],
                        multi=True,
                        placeholder="All boroughs …",
                        clearable=True,
                    ),
                ]),
                html.Div([
                    html.Div("Cuisine", className="filter-label"),
                    dcc.Dropdown(
                        id="filter-cuisine",
                        options=[{"label": c, "value": c} for c in CUISINES],
                        multi=True,
                        placeholder="All cuisines …",
                        clearable=True,
                    ),
                ]),
                html.Div([
                    html.Div("Grade", className="filter-label"),
                    dcc.Dropdown(
                        id="filter-grade",
                        options=[{"label": g, "value": g} for g in GRADES],
                        multi=True,
                        placeholder="All grades …",
                        clearable=True,
                    ),
                ]),
                html.Div([
                    html.Div("Year Range", className="filter-label"),
                    dcc.RangeSlider(
                        id="filter-year",
                        min=YEAR_MIN,
                        max=YEAR_MAX,
                        step=1,
                        value=[YEAR_MIN, YEAR_MAX],
                        marks={y: {"label": str(y), "style": {"color": THEME["text_sec"], "fontSize": "10px"}}
                               for y in range(YEAR_MIN, YEAR_MAX + 1)},
                        tooltip={"placement": "bottom", "always_visible": False},
                    ),
                ], style={"paddingTop": "4px"}),
            ]),
        ]),

        # ── KPI CARDS ───────────────────────────────────────────────────────
        html.Div(className="kpi-row", id="kpi-row", children=[
            kpi_card("Total Inspections",    KPIS_FULL["total"],        THEME["accent"]),
            kpi_card("Avg. Inspection Score",KPIS_FULL["avg_score"],    THEME["accent4"]),
            kpi_card("% Critical Violations",KPIS_FULL["pct_critical"], THEME["accent2"]),
            kpi_card("Grade A Rate",         KPIS_FULL["grade_a_pct"],  THEME["accent3"]),
        ]),

        # ── ROW 1: Boro + Cuisine ────────────────────────────────────────────
        html.Div(className="charts-grid-2", children=[
            html.Div(className="chart-card",
                     children=[dcc.Graph(id="fig-boro",    figure=fig_boro_bar(AGG_FULL),
                                         config={"displayModeBar": False})]),
            html.Div(className="chart-card",
                     children=[dcc.Graph(id="fig-cuisine", figure=fig_cuisine_bar(AGG_FULL),
                                         config={"displayModeBar": False})]),
        ]),

        # ── ROW 2: Monthly timeline (full width) ────────────────────────────
        html.Div(className="charts-grid-full chart-card",
                 children=[dcc.Graph(id="fig-monthly",
                                     figure=fig_monthly_line(AGG_FULL),
                                     config={"displayModeBar": False},
                                     style={"height": "280px"})]),

        # ── ROW 3: Score hist + Grade pie + Heatmap ─────────────────────────
        html.Div(className="charts-grid-3", children=[
            html.Div(className="chart-card",
                     children=[dcc.Graph(id="fig-score",  figure=fig_score_histogram(AGG_FULL),
                                         config={"displayModeBar": False})]),
            html.Div(className="chart-card",
                     children=[dcc.Graph(id="fig-grade",  figure=fig_grade_pie(AGG_FULL),
                                         config={"displayModeBar": False})]),
            html.Div(className="chart-card",
                     children=[dcc.Graph(id="fig-heatmap",figure=fig_heatmap(AGG_FULL),
                                         config={"displayModeBar": False})]),
        ]),

        # ── ROW 4: Violations + Critical comparison ──────────────────────────
        html.Div(className="charts-grid-2", children=[
            html.Div(className="chart-card",
                     children=[dcc.Graph(id="fig-violations",figure=fig_violation_bar(AGG_FULL),
                                         config={"displayModeBar": False})]),
            html.Div(className="chart-card",
                     children=[dcc.Graph(id="fig-critical",  figure=fig_critical_comparison(AGG_FULL),
                                         config={"displayModeBar": False})]),
        ]),

        # ── FOOTER ──────────────────────────────────────────────────────────
        html.Div(
            f"NYC DOHMH Restaurant Inspection Results · Dashboard · "
            f"Rows Loaded: {len(DF_FULL):,} · Powered by Polars + Plotly + Dash",
            className="footer"
        ),
    ]),
])


# ──────────────────────────────────────────────────────────────────────────────
# 7. CALLBACKS
# ──────────────────────────────────────────────────────────────────────────────

@app.callback(
    Output("kpi-row",       "children"),
    Output("fig-boro",      "figure"),
    Output("fig-cuisine",   "figure"),
    Output("fig-monthly",   "figure"),
    Output("fig-score",     "figure"),
    Output("fig-grade",     "figure"),
    Output("fig-violations","figure"),
    Output("fig-critical",  "figure"),
    Output("fig-heatmap",   "figure"),
    Input("filter-boro",    "value"),
    Input("filter-cuisine", "value"),
    Input("filter-grade",   "value"),
    Input("filter-year",    "value"),
)
def update_all(boros, cuisines, grades, year_range):
    # ── 1. filter in Polars (fast) ──────────────────────────────────────────
    mask = pl.lit(True)
    if boros:
        mask = mask & pl.col("BORO").is_in(boros)
    if cuisines:
        mask = mask & pl.col("CUISINE_DESCRIPTION").is_in(cuisines)
    if grades:
        mask = mask & pl.col("GRADE").is_in(grades)
    if year_range:
        mask = mask & (pl.col("YEAR") >= year_range[0]) & (pl.col("YEAR") <= year_range[1])

    df = DF_FULL.filter(mask)

    if df.shape[0] == 0:
        empty = go.Figure()
        empty.update_layout(**PLOTLY_LAYOUT,
                            title_text="No data for selected filters",
                            height=320)
        blank_kpis = [
            kpi_card("Total Inspections",    "0",     THEME["accent"]),
            kpi_card("Avg. Inspection Score","N/A",   THEME["accent4"]),
            kpi_card("% Critical Violations","0.0%",  THEME["accent2"]),
            kpi_card("Grade A Rate",         "0.0%",  THEME["accent3"]),
        ]
        return blank_kpis, empty, empty, empty, empty, empty, empty, empty, empty

    # ── 2. re-aggregate ─────────────────────────────────────────────────────
    agg  = precompute(df)
    kpis = compute_kpis(df)

    kpi_children = [
        kpi_card("Total Inspections",     kpis["total"],        THEME["accent"]),
        kpi_card("Avg. Inspection Score", kpis["avg_score"],    THEME["accent4"]),
        kpi_card("% Critical Violations", kpis["pct_critical"], THEME["accent2"]),
        kpi_card("Grade A Rate",          kpis["grade_a_pct"],  THEME["accent3"]),
    ]

    return (
        kpi_children,
        fig_boro_bar(agg),
        fig_cuisine_bar(agg),
        fig_monthly_line(agg),
        fig_score_histogram(agg),
        fig_grade_pie(agg),
        fig_violation_bar(agg),
        fig_critical_comparison(agg),
        fig_heatmap(agg),
    )


# ──────────────────────────────────────────────────────────────────────────────
# 8. STANDALONE HTML EXPORT
# ──────────────────────────────────────────────────────────────────────────────

def export_static_html(output_path: str = "dashboard_export.html"):
    """
    Build a standalone multi-panel HTML file using the full-dataset figures.
    Each chart is embedded as a self-contained Plotly HTML widget.
    """
    agg = AGG_FULL

    panels = [
        ("Inspections by Borough",        fig_boro_bar(agg)),
        ("Top 10 Cuisines",               fig_cuisine_bar(agg)),
        ("Inspections Over Time",         fig_monthly_line(agg)),
        ("Score Distribution",            fig_score_histogram(agg)),
        ("Grade Distribution",            fig_grade_pie(agg)),
        ("Top Violation Codes",           fig_violation_bar(agg)),
        ("Critical vs Non-Critical",      fig_critical_comparison(agg)),
        ("Borough × Grade Heatmap",       fig_heatmap(agg)),
    ]

    kpis = KPIS_FULL

    charts_html = ""
    include_plotlyjs = "cdn"
    for i, (title, fig) in enumerate(panels):
        fig_html = fig.to_html(
            full_html=False,
            include_plotlyjs=include_plotlyjs,
            div_id=f"chart_{i}",
            config={"displayModeBar": False},
        )
        include_plotlyjs = False  # include JS only once
        charts_html += f"""
        <div class="chart-card" style="margin-bottom:16px;padding:12px;background:#111827;
             border:1px solid #2A3347;border-radius:12px;">
          <div style="font-family:'DM Mono',monospace;font-size:12px;color:#94A3B8;
               margin-bottom:8px;text-transform:uppercase;letter-spacing:0.08em;">{title}</div>
          {fig_html}
        </div>
        """

    kpi_html = "".join([
        f"""<div style="background:#111827;border:1px solid #2A3347;border-radius:12px;
            padding:20px 24px;border-top:3px solid {c};">
          <div style="font-family:'DM Mono',monospace;font-size:26px;font-weight:500;color:{c};
               margin-bottom:4px;">{v}</div>
          <div style="font-size:10px;color:#94A3B8;text-transform:uppercase;letter-spacing:0.1em;
               font-family:'DM Mono',monospace;">{l}</div>
        </div>"""
        for l, v, c in [
            ("Total Inspections",     kpis["total"],        "#00D4FF"),
            ("Avg. Inspection Score", kpis["avg_score"],    "#FFE66D"),
            ("% Critical Violations", kpis["pct_critical"], "#FF6B6B"),
            ("Grade A Rate",          kpis["grade_a_pct"],  "#4ECDC4"),
        ]
    ])

    full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>NYC Restaurant Inspection Dashboard</title>
  <link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Outfit:wght@300;400;600&display=swap" rel="stylesheet">
  <style>
    {CUSTOM_CSS}
    body {{ margin:0; padding:0; }}
    .export-grid {{
      display:grid;
      grid-template-columns:1fr 1fr;
      gap:16px;
    }}
  </style>
</head>
<body>
  <div class="dashboard-wrapper">
    <div class="header">
      <div class="header-logo">
        <div class="header-dot"></div>
        <div>
          <div class="header-title">NYC Restaurant Inspections</div>
          <div class="header-subtitle">DOHMH · Standalone Export · Static View</div>
        </div>
      </div>
      <div class="header-badge">STATIC EXPORT · {len(DF_FULL):,} RECORDS</div>
    </div>
    <div class="main-content">
      <div class="kpi-row" style="margin-bottom:24px;">{kpi_html}</div>
      <div class="export-grid">{charts_html}</div>
      <div class="footer">NYC DOHMH Restaurant Inspection Results · Exported Dashboard · Powered by Polars + Plotly</div>
    </div>
  </div>
</body>
</html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(full_html)
    print(f"[INFO] Standalone HTML exported → {output_path}")


# ──────────────────────────────────────────────────────────────────────────────
# 9. ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    export_static_html("dashboard_export.html")

    print("[INFO] Starting Dash server on http://127.0.0.1:8050")
    print("[INFO] Press Ctrl+C to stop.")
    app.run(debug=False, host="127.0.0.1", port=8050)

[INFO] Loading data …
[INFO] Loaded 398,302 rows
[INFO] Standalone HTML exported → dashboard_export.html
[INFO] Starting Dash server on http://127.0.0.1:8050
[INFO] Press Ctrl+C to stop.
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
